In [3]:
import pandas as pd
import numpy as np

In [4]:
users = pd.read_csv("../All Users.csv")
transactions = pd.read_csv("../All transaction.csv")

In [5]:
users.head()

,User_ID,Name,Age,Join_Date
0,PP0000001,Holly Rivera,56,17-06-2025
1,PP0000002,Kevin Lopez,46,10-12-2023
2,PP0000003,Douglas Roberts,32,14-09-2024
3,PP0000004,Walter Davila,60,19-10-2023
4,PP0000005,Grace Blake,25,10-03-2025


In [6]:
transactions.head()

,Transaction_ID,Amount,User_ID,Service,Service Type,Payment_Status,Reason,Date
0,RCG_0C338474B366,926.59,PP0021371,Recharge_Bills,FASTag Recharge,Successful,Successful,09-06-2024
1,RCG_6B3B86B07A76,1211.64,PP0002388,Recharge_Bills,DTH,Successful,Successful,04-08-2024
2,RCG_767822392A0E,746.27,PP1101831,Recharge_Bills,Cable TV,Successful,Successful,19-02-2024
3,RCG_527E6AC74B11,1319.89,PP0033099,Recharge_Bills,Mobile Recharge,Successful,Successful,22-12-2024
4,RCG_6B50A8C694E1,112.44,PP1059869,Recharge_Bills,Cable TV,Successful,Successful,07-09-2024


In [7]:
users.shape

(107658, 4)

In [8]:
transactions.shape

(300000, 8)

In [9]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 107658 entries, 0 to 107657
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   User_ID    107658 non-null  str  
 1   Name       107658 non-null  str  
 2   Age        107658 non-null  int64
 3   Join_Date  107658 non-null  str  
dtypes: int64(1), str(3)
memory usage: 3.3 MB


In [10]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Transaction_ID  300000 non-null  str    
 1    Amount         300000 non-null  float64
 2   User_ID         300000 non-null  str    
 3   Service         300000 non-null  str    
 4   Service Type    300000 non-null  str    
 5   Payment_Status  300000 non-null  str    
 6   Reason          300000 non-null  str    
 7   Date            300000 non-null  str    
dtypes: float64(1), str(7)
memory usage: 18.3 MB


In [12]:
transactions["Date"].head(20)

0     09-06-2024
1     04-08-2024
2     19-02-2024
3     22-12-2024
4     07-09-2024
5     25-07-2024
6     06-08-2024
7     23-04-2024
8     25-06-2024
9     20-11-2024
10    19-08-2024
11    26-04-2024
12    17-10-2024
13    24-04-2024
14    18-12-2024
15    19-04-2024
16    24-08-2024
17    25-08-2024
18    24-03-2024
19    08-02-2024
Name: Date, dtype: str

In [13]:
transactions["Date"] = pd.to_datetime(
    transactions["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [14]:
transactions["Date"].isna().sum()

np.int64(0)

In [15]:
transactions[transactions["Date"].isna()]

,Transaction_ID,Amount,User_ID,Service,Service Type,Payment_Status,Reason,Date


In [16]:
users["Join_Date"] = pd.to_datetime(
    users["Join_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [17]:
users["Join_Date"].isna().sum()

np.int64(0)

In [19]:
reference_date = transactions["Date"].max() + pd.Timedelta(days=1)

print(reference_date)

2024-12-31 00:00:00


In [21]:
rfm = transactions.groupby("User_ID").agg({
    "Date": lambda x: (reference_date - x.max()).days,
    "Transaction_ID": "count",
    " Amount ": "sum"
})

In [23]:
rfm.columns = ["Recency", "Frequency", "Monetary"]

In [24]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=5,
    labels=[5,4,3,2,1]
)

In [25]:
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5,
    labels=[1,2,3,4,5]
)

In [26]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    q=5,
    labels=[1,2,3,4,5]
)

In [27]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(int)
    + rfm["F_Score"].astype(int)
    + rfm["M_Score"].astype(int)
)

In [30]:
def customer_segment(score):

    if score >= 13:
        return "High Value"

    elif score >= 10:
        return "Loyal"

    elif score >= 7:
        return "Regular"

    elif score >= 5:
        return "Needs Attention"

    else:
        return "Lost"

In [31]:
rfm["Customer_Segment"] = rfm["RFM_Score"].apply(customer_segment)

In [32]:
rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Customer_Segment
User_ID,,,,,,,,
PP0000001,217,1,3625.65,1,1,1,3,Lost
PP0000002,13,3,19076.22,5,3,3,11,Loyal
PP0000003,270,1,2693.60,1,1,1,3,Lost
PP0000004,7,3,23704.46,5,3,4,12,Loyal
PP0000005,145,1,998.42,2,1,1,4,Lost


In [33]:
rfm["Customer_Segment"].value_counts()

Customer_Segment
Loyal              30176
Regular            28763
High Value         16769
Needs Attention    14058
Lost               10995
Name: count, dtype: int64

In [34]:
rfm.to_csv("rfm_customer_segments.csv", index=True)